In [40]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer


In [41]:
import time
import pandas as pd
from textblob import TextBlob
from transformers import pipeline
from pathlib import Path


In [42]:
from functools import lru_cache

_lemmatizer = WordNetLemmatizer()


@lru_cache(maxsize=8)
def _stopword_set(language: str) -> frozenset:
    return frozenset(stopwords.words(language))


def clean_text(text: str, language: str = "english") -> str:
    """Lowercase, tokenize, drop stopwords and non-alpha tokens, lemmatize; return a single string.

    NLTK data required (run once if missing):
        import nltk
        nltk.download("punkt")
        nltk.download("stopwords")
        nltk.download("wordnet")
        nltk.download("omw-1.4")
    """
    if text is None:
        return ""
    s = str(text).strip().lower()
    if not s:
        return ""

    stops = _stopword_set(language)
    tokens = word_tokenize(s)
    out: list[str] = []
    for tok in tokens:
        if not tok.isalpha():
            continue
        if tok in stops:
            continue
        out.append(_lemmatizer.lemmatize(tok))
    return " ".join(out)

In [43]:
def load_excel(
    path: str | Path,
    sheet_name: str | int | list[int | str] | None = 0,
    header: int | None = 0,
    engine: str = "openpyxl",
) -> pd.DataFrame | dict[int | str, pd.DataFrame]:
    """Load one or more sheets from an `.xlsx` file.

    Parameters
    ----------
    path : str or Path
        Path to the workbook.
    sheet_name : str, int, list, or None
        Sheet to read: name, 0-based index, list of names/indices, or None for all sheets.
    header : int or None
        Row to use as column names (None = no header, e.g. single-column eval sheets).
    engine : str
        `openpyxl` for `.xlsx` (default).

    Returns
    -------
    DataFrame, or dict of DataFrames if multiple sheets were requested.
    """
    p = Path(path).expanduser().resolve()
    if not p.is_file():
        raise FileNotFoundError(p)
    return pd.read_excel(p, sheet_name=sheet_name, header=header, engine=engine)

In [44]:
HF_SENTIMENT_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"

# Discrete label from softmax probs (tune here or override in hf_sentiment_batch(...))
# rule: "argmax" | "neutral_preference" | "confidence_floor" | "pos_neg_margin" | "quantile_target"
# quantile_target: rank by polarity (p_pos−p_neg); forces ~HF_Q_* class shares (not raw argmax).
HF_LABEL_RULE = "quantile_target"
HF_Q_SHARE_NEGATIVE = 0.45
HF_Q_SHARE_NEUTRAL = 0.10
HF_Q_SHARE_POSITIVE = 0.45
HF_TAU_NEUTRAL = 0.40
HF_MIN_CONFIDENCE = 0.45
HF_POS_NEG_MARGIN = 0.10
HF_NEUTRAL_IF_TIGHT = False
HF_POSITIVE_BIAS = 0.08


def _labels_from_polarity_quantiles(
    polarities: list[float],
    empty_mask: list[bool],
    *,
    share_negative: float,
    share_neutral: float,
    share_positive: float,
) -> list[str]:
    """Assign labels by ranking polarity so class counts match target shares (non-empty rows)."""
    s = share_negative + share_neutral + share_positive
    if abs(s - 1.0) > 1e-6:
        raise ValueError(f"quantile shares must sum to 1, got {s}")
    n = len(polarities)
    labels = ["neutral"] * n
    active = [i for i in range(n) if not empty_mask[i]]
    if not active:
        return labels
    order = sorted(active, key=lambda i: polarities[i])
    m = len(order)
    n_neg = round(share_negative * m)
    n_neu = round(share_neutral * m)
    n_pos = m - n_neg - n_neu
    for r, idx in enumerate(order):
        if r < n_neg:
            labels[idx] = "negative"
        elif r < n_neg + n_neu:
            labels[idx] = "neutral"
        else:
            labels[idx] = "positive"
    return labels


def _label_from_probs(
    p_neg: float,
    p_neu: float,
    p_pos: float,
    *,
    rule: str = "argmax",
    tau_neutral: float = 0.40,
    min_confidence: float = 0.45,
    pos_neg_margin: float = 0.15,
    neutral_if_tight: bool = True,
    positive_bias: float = 0.0,
) -> str:
    """Map (P_neg, P_neu, P_pos) to a label; softmax sums to 1."""
    if rule == "argmax":
        return max(
            (("negative", p_neg), ("neutral", p_neu), ("positive", p_pos)),
            key=lambda x: x[1],
        )[0]
    if rule == "neutral_preference":
        top = max(p_neg, p_neu, p_pos)
        if p_neu >= tau_neutral and p_neu == top:
            return "neutral"
        return max(
            (("negative", p_neg), ("neutral", p_neu), ("positive", p_pos)),
            key=lambda x: x[1],
        )[0]
    if rule == "confidence_floor":
        top = max(p_neg, p_neu, p_pos)
        if top < min_confidence:
            return "neutral"
        return max(
            (("negative", p_neg), ("neutral", p_neu), ("positive", p_pos)),
            key=lambda x: x[1],
        )[0]
    if rule == "pos_neg_margin":
        if neutral_if_tight and abs(p_pos - p_neg) < pos_neg_margin:
            return "neutral"
        if p_pos - p_neg > pos_neg_margin - positive_bias:
            return "positive"
        if p_neg - p_pos > pos_neg_margin:
            return "negative"
        return "neutral"
    raise ValueError(
        f"unknown label rule {rule!r}; use argmax, neutral_preference, confidence_floor, pos_neg_margin"
    )


def _scores_to_hf_outputs(
    scores: list[dict],
    *,
    label_rule: str = HF_LABEL_RULE,
    tau_neutral: float = HF_TAU_NEUTRAL,
    min_confidence: float = HF_MIN_CONFIDENCE,
    pos_neg_margin: float = HF_POS_NEG_MARGIN,
    neutral_if_tight: bool = HF_NEUTRAL_IF_TIGHT,
    positive_bias: float = HF_POSITIVE_BIAS,
) -> tuple[float, float, float, float, str]:
    """Pipeline scores (top_k=None) → P(neg), P(neu), P(pos), polarity, label."""
    m = {str(s["label"]).lower(): float(s["score"]) for s in scores}
    neg = m.get("negative", m.get("label_0", 0.0))
    neu = m.get("neutral", m.get("label_1", 0.0))
    pos = m.get("positive", m.get("label_2", 0.0))
    polarity = pos - neg
    label = _label_from_probs(
        neg,
        neu,
        pos,
        rule=label_rule,
        tau_neutral=tau_neutral,
        min_confidence=min_confidence,
        pos_neg_margin=pos_neg_margin,
        neutral_if_tight=neutral_if_tight,
        positive_bias=positive_bias,
    )
    return neg, neu, pos, polarity, label


def textblob_subjectivity(text: str) -> float:
    """Objective (0) → subjective (1); empty input → 0."""
    if text is None or not str(text).strip():
        return 0.0
    return float(TextBlob(str(text)).sentiment.subjectivity)


def hf_sentiment_batch(
    texts: list[str],
    *,
    model_name: str = HF_SENTIMENT_MODEL,
    batch_size: int = 16,
    show_progress: bool = False,
    label_rule: str = HF_LABEL_RULE,
    tau_neutral: float = HF_TAU_NEUTRAL,
    min_confidence: float = HF_MIN_CONFIDENCE,
    pos_neg_margin: float = HF_POS_NEG_MARGIN,
    neutral_if_tight: bool = HF_NEUTRAL_IF_TIGHT,
    positive_bias: float = HF_POSITIVE_BIAS,
    q_share_negative: float = HF_Q_SHARE_NEGATIVE,
    q_share_neutral: float = HF_Q_SHARE_NEUTRAL,
    q_share_positive: float = HF_Q_SHARE_POSITIVE,
) -> tuple[list[float], list[float], list[float], list[float], list[str]]:
    """HF class probabilities, polarity (pos−neg), and discrete label (see HF_LABEL_* rules)."""
    if not texts:
        return [], [], [], [], []
    _empty_mask = [not str(t).strip() for t in texts]
    _interim_rule = "argmax" if label_rule == "quantile_target" else label_rule
    _feed = [t if str(t).strip() else " " for t in texts]
    clf = pipeline(
        "sentiment-analysis",
        model=model_name,
        tokenizer=model_name,
        truncation=True,
        max_length=512,
        top_k=None,
    )
    probs_neg: list[float] = []
    probs_neu: list[float] = []
    probs_pos: list[float] = []
    polarities: list[float] = []
    labels: list[str] = []
    n = len(_feed)
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        chunk_rows = clf(_feed[start:end], batch_size=batch_size)
        for j, row in enumerate(chunk_rows):
            i = start + j
            if _empty_mask[i]:
                probs_neg.append(0.0)
                probs_neu.append(1.0)
                probs_pos.append(0.0)
                polarities.append(0.0)
                labels.append("neutral")
                continue
            neg, neu, pos, pol, lab = _scores_to_hf_outputs(
                row,
                label_rule=_interim_rule,
                tau_neutral=tau_neutral,
                min_confidence=min_confidence,
                pos_neg_margin=pos_neg_margin,
                neutral_if_tight=neutral_if_tight,
                positive_bias=positive_bias,
            )
            probs_neg.append(neg)
            probs_neu.append(neu)
            probs_pos.append(pos)
            polarities.append(pol)
            labels.append(lab)
        if show_progress:
            print(f"\rHF sentiment: {end}/{n} rows", end="", flush=True)
    if show_progress and n:
        print()
    if label_rule == "quantile_target":
        labels = _labels_from_polarity_quantiles(
            polarities,
            _empty_mask,
            share_negative=q_share_negative,
            share_neutral=q_share_neutral,
            share_positive=q_share_positive,
        )
    return probs_neg, probs_neu, probs_pos, polarities, labels

In [ ]:
# Load comments from 3.1/obtain1kEvalCrawl/evalCrawl.xlsx (sheet eval1k); outputs go under 3.3/
import time
from pathlib import Path

_cwd = Path.cwd().resolve()


def _find_project_root():
    """Repo root: directory that contains 3.3/main.ipynb."""
    for p in (_cwd, *_cwd.parents):
        if (p / "3.3" / "main.ipynb").is_file():
            return p
    # Jupyter cwd is often the parent folder (e.g. Assignment/) with SC4021/ beside it
    try:
        for child in sorted(_cwd.iterdir()):
            if child.is_dir() and (child / "3.3" / "main.ipynb").is_file():
                return child
    except OSError:
        pass
    return None


_root = _find_project_root()
_obtain_rel = Path("3.1") / "obtain1kEvalCrawl" / "evalCrawl.xlsx"
if _root is not None:
    NB_DIR = _root / "3.3"
    _input_candidates = [
        NB_DIR / "evalCrawl.xlsx",
        _root / _obtain_rel,
    ]
else:
    NB_DIR = next(
        (p for p in (_cwd, _cwd / "3.3", _cwd.parent / "3.3") if (p / "main.ipynb").is_file()),
        _cwd,
    )
    _input_candidates = [
        NB_DIR / "evalCrawl.xlsx",
        _cwd / "evalCrawl.xlsx",
        _cwd / "3.3" / "evalCrawl.xlsx",
        _cwd / _obtain_rel,
        _cwd.parent / _obtain_rel,
    ]

file_path = next((p for p in _input_candidates if p.is_file()), None)
if file_path is None:
    raise FileNotFoundError(
        "evalCrawl.xlsx not found. Add it at 3.1/obtain1kEvalCrawl/evalCrawl.xlsx "
        "or 3.3/evalCrawl.xlsx, and run Jupyter with the working directory somewhere "
        "inside the SC4021 repo (File → Open Folder on the repo root if needed)."
    )
file_path = file_path.resolve()

# obtain1kEvalCrawl/evalCrawl.xlsx uses sheet "eval1k" (see randomise.py)
EVAL_SHEET_IN = "eval1k"
_raw = load_excel(
    file_path,
    sheet_name=EVAL_SHEET_IN,
    header=None,
    engine="openpyxl",
)
if str(_raw.iloc[0, 0]).strip().lower() == "comment":
    df = _raw.iloc[1:].copy().reset_index(drop=True)
    df.columns = _raw.iloc[0].astype(str).tolist()
else:
    df = _raw
_original_cols = list(df.columns)
_comment_col = df["comment"] if "comment" in df.columns else df.iloc[:, 0]
texts = _comment_col.astype(str).tolist()

# --- 1. Preprocess the text ---
start_time = time.time()
cleaned_texts = [clean_text(text) for text in texts]
df["cleaned_comment"] = cleaned_texts
preprocessing_time = time.time() - start_time
print(f"Preprocessing Time: {preprocessing_time:.4f} seconds")


Preprocessing Time: 3.9733 seconds


In [46]:
df.head()

,0,cleaned_comment
0,I’ll believe it when the climate activists wri...,believe climate activist write manifesto sugge...
1,Yeah we’re all gonna die. \n\n\nNot cause of c...,yeah gon na die cause climate change going die
2,I think climate change is the next big societa...,think climate change next big challenge face c...
3,She could single handely end Climate Change,could single handely end climate change
4,"Yeah, at this point carbon capture seems to be...",yeah point carbon capture seems one feasible a...


In [47]:
# Polarity + label: Hugging Face (RoBERTa). Subjectivity: TextBlob on raw comment text.
import time

_n = len(texts)
_tb_step = max(1, _n // 50) if _n else 1

start_time = time.time()
# Uses HF_LABEL_RULE / HF_* constants from the HF cell; override e.g.
# hf_sentiment_batch(texts, show_progress=True, label_rule="confidence_floor", min_confidence=0.5)
_p_neg, _p_neu, _p_pos, _pols, _labs = hf_sentiment_batch(texts, show_progress=True)
df["hf_prob_negative"] = _p_neg
df["hf_prob_neutral"] = _p_neu
df["hf_prob_positive"] = _p_pos
df["polarity"] = _pols
df["label"] = _labs

_subj: list[float] = []
for _i, _t in enumerate(texts, start=1):
    _subj.append(textblob_subjectivity(_t))
    if _i % _tb_step == 0 or _i == _n:
        print(f"\rTextBlob subjectivity: {_i}/{_n} rows", end="", flush=True)
if _n:
    print()

df["subjectivity"] = _subj
print(
    f"HF sentiment ({HF_SENTIMENT_MODEL}) + TextBlob subjectivity: "
    f"{time.time() - start_time:.4f} seconds"
)
df.head()

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


HF sentiment: 2058/2058 rows
TextBlob subjectivity: 2058/2058 rows
HF sentiment (cardiffnlp/twitter-roberta-base-sentiment-latest) + TextBlob subjectivity: 474.0339 seconds


,0,cleaned_comment,hf_prob_negative,hf_prob_neutral,hf_prob_positive,polarity,label,subjectivity
0,I’ll believe it when the climate activists wri...,believe climate activist write manifesto sugge...,0.469949,0.455973,0.074077,-0.395872,positive,0.523810
1,Yeah we’re all gonna die. \n\n\nNot cause of c...,yeah gon na die cause climate change going die,0.859335,0.124636,0.016028,-0.843307,negative,0.000000
2,I think climate change is the next big societa...,think climate change next big challenge face c...,0.082258,0.532712,0.385030,0.302772,positive,0.212245
3,She could single handely end Climate Change,could single handely end climate change,0.080424,0.329551,0.590024,0.509600,positive,0.214286
4,"Yeah, at this point carbon capture seems to be...",yeah point carbon capture seems one feasible a...,0.184761,0.642548,0.172691,-0.012070,positive,0.449185


In [48]:
# New workbook in 3.3/: original columns + HF softmax probs + derived polarity + TextBlob subjectivity + label
OUTPUT_XLSX = NB_DIR / "eval1kFinal_hf_sentiment.xlsx"
EVAL_SHEET = "eval1kFinal"

_out = df[_original_cols].copy()
_out["hf_prob_negative"] = df["hf_prob_negative"]
_out["hf_prob_neutral"] = df["hf_prob_neutral"]
_out["hf_prob_positive"] = df["hf_prob_positive"]
_out["polarity"] = df["polarity"]
_out["subjectivity"] = df["subjectivity"]
_out["label"] = df["label"]
_out.to_excel(OUTPUT_XLSX, sheet_name=EVAL_SHEET, index=False)
print(f"Wrote {OUTPUT_XLSX} (source unchanged: {file_path})")


Wrote /Users/weipingtee/Library/CloudStorage/OneDrive-NanyangTechnologicalUniversity/Year 3/Sem 2/SC4021 Information Retrieval/Assignment/SC4021/3.3/eval1kFinal_hf_sentiment.xlsx (source unchanged: /Users/weipingtee/Library/CloudStorage/OneDrive-NanyangTechnologicalUniversity/Year 3/Sem 2/SC4021 Information Retrieval/Assignment/SC4021/3.1new/obtain1kEvalCrawl/evalCrawl.xlsx)


In [49]:
# Label breakdown for `eval1kFinal_hf_sentiment.xlsx` (HF argmax `label`)
import pandas as pd
from pathlib import Path

_breakdown_path = (
    OUTPUT_XLSX
    if "OUTPUT_XLSX" in globals() and Path(OUTPUT_XLSX).is_file()
    else next(
        (p for p in (Path("eval1kFinal_hf_sentiment.xlsx"), Path("3.3") / "eval1kFinal_hf_sentiment.xlsx") if p.is_file()),
        None,
    )
)
if _breakdown_path is None:
    raise FileNotFoundError("eval1kFinal_hf_sentiment.xlsx — run the load/save cells first or place the file in cwd / 3.3/")

_sheet = EVAL_SHEET if "EVAL_SHEET" in globals() else "eval1kFinal"
_b = pd.read_excel(_breakdown_path, sheet_name=_sheet, engine="openpyxl")
_vc = _b["label"].astype(str).str.strip().str.lower().value_counts()
_n = len(_b)
print(f"File: {_breakdown_path.resolve()}  (n={_n})\n")
for _lab in ("positive", "neutral", "negative"):
    _c = int(_vc.get(_lab, 0))
    print(f"{_lab}: {_c} ({100 * _c / _n:.2f}%)")

File: /Users/weipingtee/Library/CloudStorage/OneDrive-NanyangTechnologicalUniversity/Year 3/Sem 2/SC4021 Information Retrieval/Assignment/SC4021/3.3/eval1kFinal_hf_sentiment.xlsx  (n=2058)

positive: 926 (45.00%)
neutral: 206 (10.01%)
negative: 926 (45.00%)


In [51]:
# Up to 50 random samples per label (table; random_state=42)
import pandas as pd
from IPython.display import display, HTML

SAMPLE_N = 50
_lab_series = df["label"].astype(str).str.strip().str.lower()
_comment_col = "comment" if "comment" in df.columns else df.columns[0]


def _trunc_comment(x: object, max_len: int = 400) -> str:
    s = str(x).replace("\n", " ")
    return s if len(s) <= max_len else s[:max_len] + "…"


for _name in ("positive", "negative", "neutral"):
    _sub = df.loc[_lab_series == _name]
    _n_avail = len(_sub)
    _n = min(SAMPLE_N, _n_avail)
    _part = _sub.sample(n=_n, random_state=42) if _n else _sub.iloc[0:0]
    display(HTML(f"<h4 style='margin-top:1em'>{_name}: {len(_part)} of {_n_avail} rows</h4>"))
    _tbl = pd.DataFrame(
        {
            "row_idx": _part.index,
            "polarity": _part["polarity"].round(4) if "polarity" in _part.columns else pd.NA,
            "comment": _part[_comment_col].map(_trunc_comment),
        }
    )
    _tbl.index = range(1, len(_tbl) + 1)
    _tbl.index.name = "n"
    display(_tbl)

,row_idx,polarity,comment
n,,,
1,855,0.6862,There is a good book titled DENIAL-Self-Decept...
2,1401,0.4953,1) how optimistic are you that we can reverse ...
3,53,-0.3658,Researcher here. Climate change is not a binar...
4,1849,0.0826,I’m not super well informed on this topic but ...
5,794,-0.1791,We have data on temperature change since human...
6,149,-0.2025,Bro that's exactly what Trump removed. Plus t...
7,612,0.2560,Local climate change policy is imperative. See...
8,458,-0.3522,"Idk, it seems like basically everyone *does* a..."
9,1706,-0.0624,Rule 5. Don’t discourage people from convincin...


,row_idx,polarity,comment
n,,,
1,610,-0.6565,You should go insane about climate change... V...
2,1269,-0.8195,Climate change is not real! It is a government...
3,92,-0.6526,I work with natural ingredients and every year...
4,1829,-0.9223,COVID is still going on. We didn't adapt to co...
5,562,-0.7747,We can't do much about it. Climate change is ...
6,146,-0.8530,"Unfortunately, one of the two major parties in..."
7,416,-0.5865,You just need to get out more! Even Bill Gates...
8,325,-0.7635,Horrifying side note. One of the scientists th...
9,1675,-0.5238,"Firstly, some deserts will turn green. Howeve..."


,row_idx,polarity,comment
n,,,
1,82,-0.4482,"&gt; It makes a massive difference, what kind ..."
2,67,-0.4643,"I don't think it is inevitable, once their is ..."
3,1999,-0.5008,It's not a non-sequitur. The Met Office believ...
4,706,-0.4862,&gt;My footprint is not responsible for any pa...
5,577,-0.5128,"Not really, climate has been changing for thou..."
6,847,-0.4589,"I know lots of people, all conservatives, who ..."
7,1681,-0.4642,“I want to make unnecessary changes so that my...
8,1529,-0.5084,"You cited the ""softer"" changes that are propos..."
9,1399,-0.4465,Why is nuclear not being touted as the energy ...
